# A first look at PUNCH data with the RHEF filter

This notebook is for absolute beginners — to Python, to Jupyter, and to
coronagraph images. In four short steps we will download one real image from
NASA's PUNCH mission, look at it, sharpen it with a filter called **RHEF**, and
make one nice figure.

**How to use a notebook:** click on a gray code cell and press **Shift+Enter**
to run it. Run the cells in order, top to bottom. Each one prints or draws
something when it finishes.

**What is RHEF?** The corona (the Sun's atmosphere) is millions of times
brighter near the Sun than far from it, so an ordinary image shows either a
blown-out center or a black outer field — never both. The Radial Histogram
Equalizing Filter fixes this by, in essence, adjusting the exposure separately
at every distance from the Sun. Faint wisps far out become just as visible as
the bright inner corona.

You will need **Python 3.12 or newer**. The first cell checks this for you.

## Step 0 — install the tools

This cell installs the Python packages we need (it is safe to run more than
once). The main ones: `sunpy` handles solar images, and `sunkit-image`
contains the RHEF filter.

In [ ]:
import sys
assert sys.version_info >= (3, 12), (
    f"This notebook needs Python 3.12 or newer, but this one is {sys.version.split()[0]}. "
    "Please start Jupyter from a newer Python and try again."
)

%pip install -q sunpy "sunkit-image>=0.7" punchbowl

print("All set!")

## Step 1 — download one PUNCH image

PUNCH images live in a public NASA archive. We ask for them with `Fido`,
sunpy's search tool — sort of like a search engine for solar data. Here we
request one *mosaic* (the combined view from all four PUNCH spacecraft) from
May 11, 2026. The file is about 17 MB, so give it a moment.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import punchbowl                        # teaches sunpy about PUNCH data
from concurrent.futures import ThreadPoolExecutor
from sunpy.net import Fido, attrs as a

result = Fido.search(
    a.Time("2026-05-11 12:00", "2026-05-11 12:30"),   # a half-hour window
    a.Source("PUNCH"),                                # the mission
    a.Level("3"),                                     # science-ready data
    a.punch.ProductCode("CT"), a.Instrument("M"),     # clear-sky mosaic
)

# The download runs in a helper thread; this avoids a harmless but
# scary-looking warning that Jupyter sometimes prints otherwise.
with ThreadPoolExecutor(1) as pool:
    files = pool.submit(lambda: Fido.fetch(result[0][0], path='punch_data/{file}',
                                           progress=False)).result()

print("Downloaded:", files[0])

## Step 2 — look at the raw image

First we open the file as a sunpy `Map` (an image that knows where the Sun is)
and shrink it to a comfortable working size. Then we plot it. Even with the
display stretched to show the faintest 99.5% of pixels, notice the problem:
the middle is a bright blob and the outer field is nearly black. The dark
triangle in the center is simply where no camera points.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import sunpy.map

m = sunpy.map.Map(files[0])[0]                 # [0]: the file holds several layers;
                                               # the first one is the image
m = m.resample([1024, 1024] * u.pix)           # smaller = faster, plenty for a look

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax, clip_interval=(5, 99.95) * u.percent)
ax.set_title("Raw PUNCH mosaic")
plt.show()

## Step 3 — apply the RHEF filter

This is the whole trick — one line. `rhef` figures out all its own settings,
and the result comes back with the official PUNCH colormap already attached.
Suddenly the streamers and solar-wind structure fill the entire field of view.

If you would like to experiment, `rhef` has one knob called `upsilon` that
sets how strong the effect is: try `rhef(m, upsilon=0.1)` for a gentler look
(the default is 0.35).

In [ ]:
from sunkit_image.radial import rhef

filtered = rhef(m)                             # <-- the filter

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(projection=filtered)
filtered.plot(axes=ax)
ax.set_title("Same image, after RHEF")
plt.show()

## Step 4 — one nice figure

To finish, a publication-style version of the filtered image: dark background,
rings marking the distance from the Sun in solar radii (the Sun's own radius,
about 700,000 km), and a colorbar. You do not need to understand every line —
it is all just matplotlib decoration around the same `filtered` image.

In [ ]:
import numpy as np
from matplotlib.patches import Circle

# how far the image edge is from the Sun, measured in solar radii
r_max = (m.data.shape[1] / 2) * m.scale[0].to_value(u.arcsec / u.pix) / m.rsun_obs.to_value(u.arcsec)

with plt.rc_context({"font.family": "monospace"}):
    fig, ax = plt.subplots(figsize=(7.5, 7.5), facecolor="#0c0a07")
    im = ax.imshow(filtered.data, origin="lower", extent=[-r_max, r_max, -r_max, r_max],
                   cmap="punch", vmin=0, vmax=1)
    ax.set_facecolor("black")
    for rr in (50, 100, 150):                  # distance rings, in solar radii
        ax.add_patch(Circle((0, 0), rr, fill=False, ec="#f3ead9", lw=0.5, alpha=0.25))
        ax.text(0, -rr, str(rr), color="#f3ead9", fontsize=8, alpha=0.6, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.1", fc="#0c0a07", ec="none", alpha=0.6))
    ax.set_xlim(-r_max, r_max); ax.set_ylim(-r_max, r_max)
    for s in ax.spines.values(): s.set_color("#2a2318")
    ax.tick_params(colors="#a8967a", labelsize=8)
    ax.set_xlabel(r"solar-X  [R$_\odot$]", color="#a8967a")
    ax.set_ylabel(r"solar-Y  [R$_\odot$]", color="#a8967a")
    ax.set_title(rf"PUNCH   ·   RHEF   ·   {m.date.iso[:16]} UTC",
                 color="#f3ead9", fontsize=11, pad=12)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label("RHEF equalized brightness", color="#a8967a", fontsize=9)
    cb.ax.yaxis.set_tick_params(color="#a8967a", labelsize=8)
    cb.outline.set_edgecolor("#2a2318")
    plt.setp(plt.getp(cb.ax, "yticklabels"), color="#a8967a")
plt.show()

## Where to go next

That is the whole pipeline: download, look, filter, figure. The companion
notebook `make_rhef_videos.ipynb` goes further — many frames, movies, the
`upsilon` knob in detail, and the one pitfall to avoid (`radial_bin_edges`).

_The RHEF paper:_ [Gilly & Cranmer 2025](https://link.springer.com/article/10.1007/s11207-025-02578-x) ·
_API docs:_ [`sunkit_image.radial.rhef`](https://docs.sunpy.org/projects/sunkit-image/en/stable/api/sunkit_image.radial.rhef.html)